# Natural Scene Classification

Real rocket-scene, moon and portrait images.

## Step 1: Import libraries

This cell imports image, numerical, visualization and machine-learning tools.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## Step 2: Locate the included real-image dataset

Each folder name is treated as one class.

In [ ]:
DATASET_DIR=Path("../datasets/06_natural_scenes")
print([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])

## Step 3: Load and resize images

Traditional ML needs consistent image dimensions.

In [ ]:
def load_images(path, size=(128,128)):
    images, labels = [], []
    for class_dir in sorted(path.iterdir()):
        if class_dir.is_dir():
            for fp in sorted(class_dir.glob("*.png")):
                images.append(np.array(Image.open(fp).convert("RGB").resize(size)))
                labels.append(class_dir.name)
    return np.array(images), np.array(labels)

images,labels=load_images(DATASET_DIR)
print(images.shape,labels.shape)

## Step 4: Display real image samples

This verifies the dataset.

In [ ]:
classes=sorted(set(labels))
plt.figure(figsize=(12,3))
for i,name in enumerate(classes,1):
    idx=np.where(labels==name)[0][0]
    plt.subplot(1,len(classes),i); plt.imshow(images[idx]); plt.title(name); plt.axis("off")
plt.tight_layout(); plt.show()


## Step 5: Extract features

Colour histograms capture global scene appearance.

In [ ]:
def color_hist(im,bins=16):
    f=[]
    for ch in range(3):
        h,_=np.histogram(im[:,:,ch],bins=bins,range=(0,256),density=True); f.extend(h)
    return np.array(f)
features=np.array([color_hist(im) for im in images])
print(features.shape)


## Step 6: Split train and test sets

The test set remains unseen during training.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(features,labels,test_size=.25,random_state=42,stratify=labels)
print("Train:",len(X_train),"Test:",len(X_test))


## Step 7: Train the model

Random Forest performs multi-class classification.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(n_estimators=250,random_state=42,class_weight="balanced")
model.fit(X_train,y_train)


## Step 8: Evaluate

We use accuracy, classification report and confusion matrix.

In [ ]:
pred=model.predict(X_test)
print("Accuracy:",round(accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred,xticks_rotation=45)
plt.tight_layout(); plt.show()


## Conclusion

Workflow: real images → preprocessing → feature extraction → traditional ML → evaluation.

# Test One Single Natural-Scene Image

This section classifies one image as landscape, moon or space.

The notebook's combined `scene_features()` function is used again so the new image receives exactly the same feature representation as the training images.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

single_image_path = Path(
    "../datasets/06_natural_scenes/moon/000_original.png"
)

single_image = Image.open(single_image_path).convert("RGB")
single_image = single_image.resize((128, 128))
single_image_array = np.array(single_image)

single_features = scene_features(single_image_array)
single_features_2d = single_features.reshape(1, -1)

predicted_class = model.predict(single_features_2d)[0]

print("Predicted scene:", predicted_class)

plt.figure(figsize=(5, 5))
plt.imshow(single_image_array)
plt.title(f"Predicted Scene: {predicted_class}")
plt.axis("off")
plt.show()


## Test one single scene image

This section uses the same preprocessing and feature-extraction function used during training. It checks the file path, creates one two-dimensional model input row, predicts the class, and displays class probabilities when the trained model supports `predict_proba()`.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

single_image_path = Path("../datasets/06_natural_scenes/moon/000_original.png")

if not single_image_path.exists():
    raise FileNotFoundError(
        f"Image not found: {single_image_path.resolve()}"
    )

single_image = Image.open(single_image_path).convert("RGB")
single_image = single_image.resize((128, 128))
single_image_array = np.array(single_image)

single_features = scene_features(single_image_array)
single_features_2d = np.asarray(single_features).reshape(1, -1)

predicted_class = model.predict(single_features_2d)[0]

print("Image shape:", single_image_array.shape)
print("Feature shape:", np.asarray(single_features).shape)
print("Model input shape:", single_features_2d.shape)
print("Predicted class:", predicted_class)

if hasattr(model, "predict_proba"):
    probabilities = model.predict_proba(single_features_2d)[0]
    class_labels = model.classes_

    print("\nClass probabilities:")
    for label, probability in zip(class_labels, probabilities):
        print(f"{label}: {probability * 100:.2f}%")
else:
    print("\nThis trained model does not provide predict_proba().")
    if hasattr(model, "decision_function"):
        print("Decision score:", model.decision_function(single_features_2d))

plt.figure(figsize=(5, 5))
plt.imshow(single_image_array, cmap="gray" if single_image_array.ndim == 2 else None)
plt.title(f"Predicted: {predicted_class}")
plt.axis("off")
plt.show()
